# Certifiable localization in $SE(3)$

This notebook solves the same deterministic 20-pose problem three ways: local Gauss–Newton optimization, a monolithic semidefinite relaxation, and a chordally decomposed semidefinite relaxation.

The g2o file stores conventional ground-truth poses $wTk$. The certifiable formulation instead optimizes $kTw=(wTk)^{-1}$ so its point-correspondence and left-between residuals are affine in homogeneous pose-matrix entries.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/CertifiableLocalizationExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
import plotly.graph_objects as go

## Convert the loaded measurements

For measured bearing $u$, range $r$, and Unit3 tangent basis $B$,

$$
\widetilde{kP}=ru,\qquad
DkP_{/\mathrm{bearingRange}}=\begin{bmatrix}rB & u\end{bmatrix},\qquad
\Sigma_{kP}=DkP_{/\mathrm{bearingRange}}\Sigma_{br}DkP_{/\mathrm{bearingRange}}^{\mathsf T}.
$$

`KnownLandmarkFactor2Pose3` uses the residual $kTw\,wL-\widetilde{kP}$. For consecutive inverse states, the loaded relative measurement satisfies $iTw=iTj\,jTw$, which is exactly the convention of `FrobeniusLeftBetweenFactorPose3`. These two factors provide an exact D=1 QCQP representation of their nonlinear objectives.

In [3]:
data_file = gtsam.findExampleDataFile(
    "known_landmark_localization_20.g2o"
)
source_graph, dataset_values = gtsam.readG2o(data_file, is3D=True)

graph = gtsam.NonlinearFactorGraph()
pose_keys, landmark_keys = set(), set()
kPCovariances = []
for index in range(source_graph.size()):
    factor = source_graph.at(index)
    keys = tuple(factor.keys())
    if isinstance(factor, gtsam.BearingRangeFactor3D):
        k, l = keys
        measured_kBearingRange = factor.measured()
        kRange = measured_kBearingRange.range()
        kBearing = measured_kBearingRange.bearing()
        measured_kP = kRange * kBearing.unitVector()
        DkP_dbearingRange = np.column_stack(
            (kRange * kBearing.basis(), kBearing.unitVector())
        )
        kPCovariance = (
            DkP_dbearingRange
            @ factor.noiseModel().covariance()
            @ DkP_dbearingRange.T
        )
        graph.add(
            gtsam.KnownLandmarkFactor2Pose3(
                k,
                dataset_values.atPoint3(l),
                measured_kP,
                gtsam.noiseModel.Gaussian.Covariance(kPCovariance),
            )
        )
        pose_keys.add(k)
        landmark_keys.add(l)
        kPCovariances.append(kPCovariance)
    elif isinstance(factor, gtsam.BetweenFactorPose3):
        i, j = keys
        graph.add(
            gtsam.FrobeniusLeftBetweenFactorPose3(
                i, j, factor.measured(), factor.noiseModel()
            )
        )
        pose_keys.update(keys)
    else:
        raise TypeError(f"Unexpected factor type: {type(factor).__name__}")

pose_keys, landmark_keys = sorted(pose_keys), sorted(landmark_keys)
eigenvalues = np.linalg.eigvalsh(kPCovariances[0])
anisotropicity = np.sqrt(eigenvalues[-1] / eigenvalues[0])
print(f"certifiable factors: {graph.size()}")
print(f"inverse pose states:     {len(pose_keys)}")
print(f"sqrt covariance condition number: {anisotropicity:.1f}")
assert np.isclose(anisotropicity, 10.0)

certifiable factors: 99
inverse pose states:     20
sqrt covariance condition number: 10.0


## Local optimization over $kTw$

Loaded $wTk$ values are inverted before insertion. Because the measurements are noisy, the maximum-likelihood estimate can have lower objective than the generating trajectory while remaining close to it.

In [4]:
perturbation = np.array([0.02, -0.015, 0.01, 0.08, -0.05, 0.06])
ground_truth_kTws = gtsam.Values()
initial_kTws = gtsam.Values()
for index, k in enumerate(pose_keys):
    kTw = dataset_values.atPose3(k).inverse()
    ground_truth_kTws.insert(k, kTw)
    initial_kTws.insert(k, kTw.retract((index + 1) * perturbation))

parameters = gtsam.GaussNewtonParams()
parameters.setMaxIterations(100)
parameters.setRelativeErrorTol(1e-12)
gauss_newton_kTws = gtsam.GaussNewtonOptimizer(
    graph, initial_kTws, parameters
).optimize()

def pose_errors(actual_kTws):
    return np.array([
        np.linalg.norm(
            ground_truth_kTws.atPose3(k).localCoordinates(actual_kTws.atPose3(k))
        )
        for k in pose_keys
    ])

ground_truth_error = graph.error(ground_truth_kTws)
initial_error = graph.error(initial_kTws)
gauss_newton_error = graph.error(gauss_newton_kTws)
gauss_newton_pose_errors = pose_errors(gauss_newton_kTws)

print(f"ground-truth error: {ground_truth_error:.3f}")
print(f"initial error:      {initial_error:.3f}")
print(f"Gauss-Newton error: {gauss_newton_error:.3f}")
print(f"maximum pose error: {gauss_newton_pose_errors.max():.4f} m")

assert gauss_newton_error < initial_error
assert gauss_newton_error < ground_truth_error
assert gauss_newton_pose_errors.max() < 0.05

ground-truth error: 620.180
initial error:      964140.401
Gauss-Newton error: 556.769
maximum pose error: 0.0207 m


## Monolithic and chordal semidefinite relaxations

`QcqpProblem` converts both certifiable factors exactly. The monolithic solver uses one positive-semidefinite cone; the chordal solver uses a METIS elimination ordering and overlapping clique cones. The optional MOSEK wrappers recover one homogeneous QCQP vector per pose and report the largest-to-second-largest eigenvalue ratio of each recovered block.

In [5]:
qcqp = gtsam.QcqpProblem(graph, 1)
sdp_results_kTws = {}
sdp_summaries = {}

if not hasattr(gtsam, "MosekMonolithicSDP"):
    print("This GTSAM build does not include the optional MOSEK backend.")
else:
    solvers = {
        "monolithic SDP": gtsam.MosekMonolithicSDP(qcqp),
        "chordal SDP": gtsam.MosekChordalSDP(
            qcqp, gtsam.ChordalOrderingType.Metis
        ),
    }
    for name, solver in solvers.items():
        if not solver.solve(
            {"intpntCoTolRelGap": 1e-10, "optimizerMaxTime": 600.0}
        ):
            raise RuntimeError(f"{name} did not return a readable solution")

        recovered_kTws = gtsam.extractQcqpValuesPose3(solver.qcqpValues())
        recovered_pose_errors = pose_errors(recovered_kTws)
        recoveredEVRs = np.asarray(solver.variableEVRs())
        sdp_results_kTws[name] = recovered_kTws
        sdp_summaries[name] = {
            "objective": solver.objectiveValue(),
            "feasible_error": graph.error(recovered_kTws),
            "time": solver.solveTimeSeconds(),
            "evrs": recoveredEVRs,
            "pose_errors": recovered_pose_errors,
        }
        print(
            f"{name:15s}: objective={solver.objectiveValue():.3f}, "
            f"feasible error={graph.error(recovered_kTws):.3f}, "
            f"time={solver.solveTimeSeconds():.3f} s, "
            f"min EVR={recoveredEVRs.min():.3e}, "
            f"max pose error={recovered_pose_errors.max():.4f} m"
        )

        assert recovered_pose_errors.max() < 0.05
        assert np.all(recoveredEVRs >= 1e5)

monolithic SDP : objective=556.769, feasible error=556.769, time=0.844 s, min EVR=1.024e+10, max pose error=0.0207 m


chordal SDP    : objective=556.770, feasible error=556.769, time=0.546 s, min EVR=1.630e+08, max pose error=0.0207 m


## Compare all results in the world frame

Every optimized state is $kTw$. We invert it before plotting so all trajectories and landmarks are expressed in the world frame.

In [6]:
def world_translations(kTws):
    return np.vstack([
        kTws.atPose3(k).inverse().translation() for k in pose_keys
    ])

ground_truth_wPs = np.vstack([
    dataset_values.atPose3(k).translation() for k in pose_keys
])
initial_wPs = world_translations(initial_kTws)
gauss_newton_wPs = world_translations(gauss_newton_kTws)
wLs = np.vstack([dataset_values.atPoint3(l) for l in landmark_keys])

fig = go.Figure()
fig.add_scatter(
    x=initial_wPs[:, 0], y=initial_wPs[:, 1], mode="lines+markers",
    line={"dash": "dash"}, name="initial kTw (shown as wTk)",
)
fig.add_scatter(
    x=ground_truth_wPs[:, 0], y=ground_truth_wPs[:, 1],
    mode="lines+markers", line={"color": "black"}, name="ground truth wTk",
)
fig.add_scatter(
    x=gauss_newton_wPs[:, 0], y=gauss_newton_wPs[:, 1],
    mode="lines+markers", marker={"symbol": "x"}, name="Gauss-Newton",
)
for name, recovered_kTws in sdp_results_kTws.items():
    recovered_wPs = world_translations(recovered_kTws)
    fig.add_scatter(
        x=recovered_wPs[:, 0], y=recovered_wPs[:, 1],
        mode="lines+markers", name=name,
    )
fig.add_scatter(
    x=wLs[:, 0], y=wLs[:, 1], mode="markers",
    marker={"symbol": "star", "size": 12}, name="known wL",
)
fig.update_layout(
    title="Certifiable localization: local and SDP solutions",
    xaxis_title="world x [m]", yaxis_title="world y [m]",
    template="plotly_white", width=780, height=540,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

## Reading the result

`KnownLandmarkFactor2Pose3` and `FrobeniusLeftBetweenFactorPose3` use the inverse $kTw$ convention because it exposes an exact quadratic objective in homogeneous pose-matrix entries. Large EVRs indicate numerically rank-one SDP blocks, so the recovered QCQP vectors define feasible poses. All reported trajectory comparisons invert those states back to conventional $wTk$.